# Seeing the intensity on a surface

The other spatio-temporal notebooks draw *where the events landed*. This one draws
the field they landed in: the surface itself, coloured by
$\lambda(t, x \mid H_t)$, animated over time. Press play on any figure below and
drag it around -- the excitation blooms where an event lands and decays away
between them.

Four surfaces, and they are **not equally honest**. Two are drawn exactly and two
are immersed, which distorts the distances the colour is built from. Each figure
says which it is in its own caption, and the sections below say why.

This notebook executes on every documentation build. It is budgeted to finish in
well under a minute, which is why the event counts are small and why the
projective plane borrows the sphere's realisation rather than simulating its own.

In [ ]:
import numpy as np
from IPython.display import HTML

import hawkes_package as hp
from hawkes_package.spatio_temporal import FundamentalDomain, Sphere, Torus2D
from hawkes_package.viz import build_figure, embed, intensity_frames

# A 36x36 grid over 24 frames. The page carries a colour per vertex per frame, so
# this is what keeps each figure around half a megabyte instead of one and a half.
GRID, FRAMES = (36, 36), 24


def show(figure):
    # myst-nb renders a `text/html` output verbatim, but plotly's own display path
    # emits `application/vnd.plotly.v1+json`, which myst-nb skips with a warning --
    # and this build turns warnings into errors, so the figure would vanish and take
    # the docs job with it. Going through `to_html` is what keeps it both
    # interactive and buildable. "cdn" rather than "inline" because the library is
    # 5 MB against the figure's 0.5 MB.
    return HTML(figure.to_html(include_plotlyjs="cdn", full_html=False))


def draw(process, **kwargs):
    """Simulate-free: build the frames for a process that already has a record."""
    horizon = float(process.events[0, -1])
    frames = intensity_frames(process, np.linspace(0.0, horizon, FRAMES), resolution=GRID, **kwargs)
    return frames, show(build_figure(process, frames))

## A process on a flat torus

The same `SpatioTemporalHawkesProcess` as anywhere else, with a `Torus2D` for a
domain. Eight events, because each one costs a quadrature sweep over the whole
domain plus a Metropolis chain for its location, and this notebook runs on every
docs build.

In [ ]:
torus = Torus2D(3.0, 5.0)
on_torus = hp.SpatioTemporalHawkesProcess(
    base=lambda x: 0.25,
    spatial=lambda d: 1.2 * np.exp(-2.5 * d),
    temporal=lambda s: 1.8 * np.exp(-1.2 * s),
    domain=torus,
    monotone_temporal_kernel=True,  # this kernel decays from t=0, so the bound is its current value
    rng=17,
)
on_torus.simulate(8)

torus_frames, figure = draw(on_torus)
print(torus_frames.summary())
figure

## What the donut is not

The picture above is an **immersion, not an isometry**. The flat torus admits no
isometric $C^2$ embedding in $\mathbb{R}^3$ at all, so something has to give: the
outer rim of the donut is stretched and the inner rim compressed. The geodesic
distances that drive the intensity are *not* the distances you measure on screen.

That is a property of the surface, not a shortcut taken here, and every embedding
carries it in writing so a caption can repeat it.

In [ ]:
for domain in (torus, Sphere()):
    surface = embed(domain)
    print(f"{surface.name:12s} isometric={surface.isometric}")
    print(f"  {surface.note}\n")

## The sphere, drawn as itself

The one surface with nothing to apologise for. `SphericalPlane.lift` -- the same
map every geometric predicate in the package runs through -- *is* the isometric
embedding, so `viz` reuses it rather than writing a second copy of the spherical
coordinates. What you measure on screen is what the kernel measured.

Note the chart runs to both endpoints in longitude. The $\varphi = -\pi$ and
$\varphi = +\pi$ columns name the same points of the sphere, and including both is
what closes the seam instead of leaving a crack down the picture.

In [ ]:
on_sphere = hp.SpatioTemporalHawkesProcess(
    base=lambda x: 0.25,
    spatial=lambda d: 1.2 * np.exp(-2.5 * d),
    temporal=lambda s: 1.8 * np.exp(-1.2 * s),
    domain=Sphere(),
    monotone_temporal_kernel=True,
    rng=23,
)
on_sphere.simulate(8)

sphere_frames, figure = draw(on_sphere)
figure

## The Klein bottle, and the flip

A rectangle again, but glued with one sign changed: the translation
$(x, y) \mapsto (x + w, y)$ stays, and the other pairing becomes a **glide
reflection**, $(x, y) \mapsto (-x, y + h)$. That single sign is the whole
difference between the torus and the Klein bottle, and it makes the surface
non-orientable.

Drawn here as the figure-8 immersion, which self-intersects along a circle. It has
to: the Klein bottle embeds in no three-space at all. `viz` reads the glide off
the domain's own pairing matrices rather than assuming which axis carries it --
line the immersion's base circle up with the wrong one and the picture still
renders, still looks like a Klein bottle, and tears the field across a seam.

In [ ]:
bottle = FundamentalDomain.klein_bottle(3.0, 5.0)
on_bottle = hp.SpatioTemporalHawkesProcess(
    base=lambda x: 0.25,
    spatial=lambda d: 1.2 * np.exp(-2.5 * d),
    temporal=lambda s: 1.8 * np.exp(-1.2 * s),
    domain=bottle,
    monotone_temporal_kernel=True,
    rng=11,
)
# Six rather than eight: a glued surface reduces both points through the deck group
# before it can measure a distance, which costs about twenty times a torus does.
on_bottle.simulate(6)

bottle_frames, figure = draw(on_bottle)
figure

The flip is in the **numbers**, not only in the geometry. The grid spans the
fundamental rectangle and every node was folded through the domain's own `wrap`,
so the last column is the wrapped image of the first -- *reversed*. If the fold
were skipped or the axes transposed, the two would come out merely equal, and the
surface would be a torus.

In [ ]:
edge, opposite = bottle_frames.values[:, :, -1], bottle_frames.values[:, ::-1, 0]
print("glued edges agree after the flip:", np.allclose(edge, opposite, atol=1e-12, rtol=0))
print("largest disagreement:", np.abs(edge - opposite).max())

# Without the reversal they do not match: that is the glide, made arithmetic.
print("...and without it:", np.abs(edge - bottle_frames.values[:, :, 0]).max())

## The projective plane, on the sphere's own events

$\mathbb{RP}^2$ is the sphere with antipodes identified. Its fundamental polygon
is a hemisphere, but the picture below is the **whole sphere**: the covering map
$S^2 \to \mathbb{RP}^2$ is a local isometry, so the double cover is exact, where
any immersion of $\mathbb{RP}^2$ itself -- a Boy surface, say -- would distort
every distance the colour encodes.

Rather than simulate, this reuses the sphere's realisation from above, folded into
the polygon with `wrap`. Partly for the time budget, since a quotient of the sphere
reduces through its deck group on every distance call. Mostly because it isolates
the thing worth seeing: **the same events, and the only change is the geometry**.

In [ ]:
plane = FundamentalDomain.projective_plane()
on_plane = hp.SpatioTemporalHawkesProcess(
    base=lambda x: 0.25,
    spatial=lambda d: 1.2 * np.exp(-2.5 * d),
    temporal=lambda s: 1.8 * np.exp(-1.2 * s),
    domain=plane,
    monotone_temporal_kernel=True,
    rng=0,
)

# `process.events = ...` is how a realisation is seeded from events you did not
# simulate -- the same door `inference` binds a `History` through. The locations are
# folded first, so every one of them names a point of the polygon.
borrowed = np.array(on_sphere.events)
borrowed[1:, :] = np.array([plane.wrap(x) for x in borrowed[1:, :].T]).T
on_plane.events = borrowed
on_plane.n_simulated = borrowed.shape[1]

print("every folded event lies in the polygon:", all(plane.contains(x) for x in borrowed[1:, :].T))

plane_frames, figure = draw(on_plane)
figure

The colouring comes out **antipodally symmetric**, and that symmetry *is* the
identification made visible: opposite points of the sphere are one point of the
surface, so they cannot carry different intensities.

Checked through the chart rather than through grid indices, because the antipode
of $(\theta, \varphi)$ is $(\pi - \theta, \varphi \mp \pi)$ -- which is not a
reversal of the grid.

The fold is doing real work here. `FundamentalDomain.distance` reduces both of its
arguments on its own, so the excitation term would be invariant regardless; `base`
is not, because it is handed the raw chart point. With a background that varies
across the chart, skipping the fold tears the field along the equator.

In [ ]:
rng = np.random.default_rng(3)
points = np.column_stack([rng.uniform(0.0, np.pi, 200), rng.uniform(-np.pi, np.pi, 200)])
longitude = points[:, 1]
antipodes = np.column_stack(
    [np.pi - points[:, 0], np.where(longitude > 0, longitude - np.pi, longitude + np.pi)]
)

here = np.array([on_plane.intensity(1.0, plane.wrap(p)) for p in points])
there = np.array([on_plane.intensity(1.0, plane.wrap(p)) for p in antipodes])
print("largest antipodal disagreement:", np.abs(here - there).max())

## Where it stops, and why

Four surfaces, not all of them. A genus-2 surface is hyperbolic, and by Hilbert's
theorem no complete surface of constant negative curvature embeds isometrically in
three-space -- so there is no honest solid to draw. `embed` refuses rather than
drawing a dishonest one, which is the same choice the simulator makes when it
would otherwise have to guess.

Such a surface still *simulates* perfectly well; see
[Compact surfaces from fundamental domains](surfaces.ipynb). It is the picture
that is unavailable, not the mathematics.

In [ ]:
try:
    embed(FundamentalDomain.genus(2))
except ValueError as exc:
    print(exc)

## Where to go next

- [Visualization](../visualization.md) -- the one-call `animate_intensity`, which
  writes a standalone page instead of a figure, plus what each surface costs and
  how big the output gets.
- [Spatio-temporal Hawkes processes](spatio_temporal.ipynb) -- the same intensity
  as a flat field, and the two kernels it is built from.
- [Compact surfaces from fundamental domains](surfaces.ipynb) -- where these
  surfaces come from, and the ones no picture reaches.